## Adding LOFTEE as an indel tool

LOFTEE labels loss-of-function indels as HC (high-confidence) or LC (low-confidence)

The file has one row per transcript (36,664 rows, 7,822 variants), so we collapse to the MANE Select transcript for one row per variant. HC/LC is turned into a score: HC = 1, LC = 0. LOFTEE is almost all HC, so it acts as a near-binary "damaging" flag rather than a graded score.

In [17]:
lof = pd.read_csv("sites_biallelic_exome.indels.vep112_loftee.plof_calls.tsv",
                  sep="\t", low_memory=False)
print("rows:", len(lof), "| unique variants:", lof["variant_id"].nunique())
print(lof.columns.tolist)

# 1. One row per variant: keep the canonical transcript
lof1 = lof[lof["mane_select"] != "."].drop_duplicates("variant_id")
print("after collapsing to canonical:", len(lof1))

# 2. LOFTEE call distribution
print(lof1["loftee_lof"].value_counts(dropna=False))

# 3. Turn the call into a numeric "tool score": HC = 1 (confident LoF), LC = 0
lof1["loftee_score"] = (lof1["loftee_lof"] == "HC").astype(int)
print(lof1[['variant_id', 'chrom', 'pos', 'ref', 'alt', 'vcf_id', 'filter', 'gene',
       'ensembl_gene_id', 'transcript', 'biotype', 'consequence', 'impact',
       'hgvsc', 'hgvsp', 'canonical', 'mane_select', 'existing_variation',
       'loftee_lof']].head(10).to_string())

rows: 36664 | unique variants: 7822
<bound method IndexOpsMixin.tolist of Index(['variant_id', 'chrom', 'pos', 'ref', 'alt', 'vcf_id', 'filter', 'gene',
       'ensembl_gene_id', 'transcript', 'biotype', 'consequence', 'impact',
       'hgvsc', 'hgvsp', 'canonical', 'mane_select', 'existing_variation',
       'loftee_lof', 'loftee_filter', 'loftee_flags', 'loftee_info'],
      dtype='str')>
after collapsing to canonical: 7056
loftee_lof
HC    6814
LC     242
Name: count, dtype: int64
                  variant_id chrom       pos  ref       alt vcf_id                   filter  gene  ensembl_gene_id       transcript         biotype         consequence impact hgvsc hgvsp canonical  mane_select existing_variation loftee_lof
0         chr1:17022615:CA:C  chr1  17022615   CA         C      .                     PASS  SDHB  ENSG00000117118  ENST00000375499  protein_coding  frameshift_variant   HIGH     .     .       YES  NM_003000.3                  .         HC
7   chr1:17022652:A:ACAGGCGG  c

### OncoKB indel labels

The OncoKB file has 9,725 indel rows. Their labels are heavily one-sided: about 4,376 oncogenic (Oncogenic + Likely Oncogenic) but only 31 Likely Neutral, plus 5,312 Unknown. So this will be a recall/coverage analysis, not a discrimination one, since there are too few negatives.

In [ ]:
onco = pd.read_csv("exonic_toolscores_oncokb_apicall.txt", sep="\t", low_memory=False)

# how are indels written in the OncoKB (ANNOVAR) file?
onco_indels = onco[(onco["Ref"] == "-") | (onco["Alt"] == "-")]
print("OncoKB indel rows:", len(onco_indels))
print(onco_indels[["Chr","Start","End","Ref","Alt","Gene.refGeneWithVer",
                   "ExonicFunc.refGeneWithVer","ONCOGENIC"]].head(10).to_string())

# how many of those indels actually have a usable OncoKB label?
print("\nONCOGENIC labels among indels:")
print(onco_indels["ONCOGENIC"].value_counts())

OncoKB indel rows: 9725
     Chr     Start       End Ref                    Alt Gene.refGeneWithVer ExonicFunc.refGeneWithVer         ONCOGENIC
4   chr1  17022616  17022616   A                      -                SDHB       frameshift deletion  Likely Oncogenic
19  chr1  17022652  17022652   -                CAGGCGG                SDHB      frameshift insertion  Likely Oncogenic
21  chr1  17022654  17022654   A                      -                SDHB       frameshift deletion  Likely Oncogenic
22  chr1  17022655  17022655   -                GGGTCGT                SDHB      frameshift insertion  Likely Oncogenic
27  chr1  17022657  17022657   -              GGGTCGTTA                SDHB   nonframeshift insertion           Unknown
28  chr1  17022657  17022657   -                 TCGTTC                SDHB                  stopgain           Unknown
30  chr1  17022659  17022659   -  GTTGTCCAAACGTTCGTTGGT                SDHB   nonframeshift insertion           Unknown
31  chr1  170226

### Merge LOFTEE onto the OncoKB indels

LOFTEE writes variants VCF-style (CA>C) and OncoKB uses ANNOVAR-style (A>-), so the same indel is spelled differently. `vcf_to_annovar` rewrites each LOFTEE variant into ANNOVAR form (verified by hand against real matching pairs, deletions and insertions both), then we merge on Chr/Start/Ref/Alt. 6,781 of 9,725 OncoKB indels matched a LOFTEE call.

In [ ]:
# Convert LOFTEE's VCF-style variants to ANNOVAR-style keys so they match the OncoKB file
def vcf_to_annovar(pos, ref, alt):
    pos, ref, alt = int(pos), str(ref), str(alt)
    if len(ref) > len(alt):                 # deletion
        return pos + len(alt), ref[len(alt):], "-"
    elif len(alt) > len(ref):               # insertion
        return pos + len(ref) - 1, "-", alt[len(ref):]
    return pos, ref, alt                     # (shouldn't happen in an indel file)

conv = lof1.apply(lambda r: vcf_to_annovar(r["pos"], r["ref"], r["alt"]), axis=1)
lof1["Chr"]   = lof1["chrom"]
lof1[["Start","Ref","Alt"]] = pd.DataFrame(conv.tolist(), index=lof1.index)

# Merge LOFTEE score onto the OncoKB indel variants
onco_indels = onco[(onco["Ref"] == "-") | (onco["Alt"] == "-")].copy()
merged = onco_indels.merge(
    lof1[["Chr","Start","Ref","Alt","loftee_lof","loftee_score"]],
    on=["Chr","Start","Ref","Alt"], how="left")

print("OncoKB indels:", len(onco_indels))
print("matched to a LOFTEE call:", merged["loftee_lof"].notna().sum())

OncoKB indels: 9725
matched to a LOFTEE call: 6781


### LOFTEE recall on oncogenic indels

Of 4,376 OncoKB-oncogenic indels, LOFTEE flags 4,055 as high-confidence loss-of-function: recall 0.927. (103 matched but were low-confidence; 218 did not match, i.e. in-frame indels outside LOFTEE's scope.)

The missense tools scored none of these indels. LOFTEE recovers about 93 percent of them, directly covering the loss-of-function blind spot the missense panel missed. This is the main result.

In [18]:
# positives = OncoKB oncogenic indels
onco_pos = merged[merged["ONCOGENIC"].isin(["Oncogenic","Likely Oncogenic"])]

print("Oncogenic indels:", len(onco_pos))
print("  matched to a LOFTEE call:", onco_pos["loftee_lof"].notna().sum())
print("  flagged HC (damaging) by LOFTEE:", (onco_pos["loftee_lof"] == "HC").sum())

# LOFTEE recall: of oncogenic indels, fraction it flags as damaging
# (unmatched or LC both count as "not caught")
recall = (onco_pos["loftee_lof"] == "HC").mean()
print(f"\nLOFTEE recall on oncogenic indels: {recall:.3f}")

Oncogenic indels: 4376
  matched to a LOFTEE call: 4158
  flagged HC (damaging) by LOFTEE: 4055

LOFTEE recall on oncogenic indels: 0.927


### Specificity is not evaluable here

Of the 31 neutral indels, all 31 did not match LOFTEE (NaN). 

In [20]:
onco_neg = merged[merged["ONCOGENIC"] == "Likely Neutral"]
print("Neutral indels:", len(onco_neg))
print("  flagged HC by LOFTEE:", (onco_neg["loftee_lof"] == "HC").sum())
print("  specificity (correctly NOT flagged):",
      round((onco_neg["loftee_lof"] != "HC").mean(), 3))
print(onco_neg["loftee_lof"].value_counts(dropna=False))

Neutral indels: 31
  flagged HC by LOFTEE: 0
  specificity (correctly NOT flagged): 1.0
loftee_lof
NaN    31
Name: count, dtype: int64


### ClinVar has no indels to add negatives

To get benign loss-of-function indels for a real specificity estimate, we checked the ClinVar file. It is missense-only: 0 indel rows out of 102,603. So it cannot supply benign indels, and specificity would require a separate ClinVar export that includes indels. 

In [21]:
clinvar = pd.read_csv("hersh_exome_biallelic_annovar_annotated.hg38_multianno.missense_ref_alt_chrom_position_gene_clinvar.tsv",
                      sep="\t", low_memory=False)

# indels have ref/alt of different lengths (or a "-")
r = clinvar["Ref"].astype(str)
a = clinvar["Alt"].astype(str)
is_indel = (r == "-") | (a == "-") | (r.str.len() != a.str.len())

print("total ClinVar rows:", len(clinvar))
print("indel rows:", is_indel.sum())
print("\nvariant types present:")
print(clinvar.loc[is_indel, "ClinVar_CLNSIG"].value_counts().head())

total ClinVar rows: 102603
indel rows: 0

variant types present:
Series([], Name: count, dtype: int64)
